# Cleaning Census 2011 (India) Table C-08
**Educational Level by Age and Sex, for Population Age 7 and Above**

Source file: `DDW-0000C-08.xlsx` — a standard data-product export from the
Registrar General & Census Commissioner of India.

Raw government census exports like this one are almost never analysis-ready.
This notebook walks through the specific problems in *this* file and fixes
each one, one at a time, so the cleaning logic is auditable rather than a
black box.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np

SRC = 'DDW-0000C-08.xlsx'
SHEET = 'C-08'

pd.set_option('display.max_columns', 8)


## 2. Load the raw sheet and see what we're dealing with

We load with `header=None` first, on purpose — Excel exports like this one
use a multi-row, merged-cell header that pandas cannot parse automatically.
Letting pandas guess a header row would silently produce garbage column
names.

In [1]:
raw = pd.read_excel(SRC, sheet_name=SHEET, header=None)
print('Raw shape:', raw.shape)
print(raw.iloc[:9, :6].to_string())


Raw shape: (3139, 45)
       0      1       2          3       4                                                                            5
0    NaN    NaN     NaN        NaN     NaN  C-8  EDUCATIONAL LEVEL BY AGE AND SEX FOR POPULATION AGE 7 AND ABOVE - 2011
1  Table  State  Distt.  Area Name  Total/                                                                    Age-group
2   Name   Code    Code        NaN  Rural/                                                                          NaN
3    NaN    NaN     NaN        NaN  Urban/                                                                          NaN
4    NaN    NaN     NaN        NaN     NaN                                                                          NaN
5    NaN    NaN     NaN        NaN     NaN                                                                            1
6    NaN    NaN     NaN        NaN     NaN                                                                          NaN
7  C2308     00   

### Problems found in the raw sheet

1. **Row 0** is just the table title (`"C-8  EDUCATIONAL LEVEL BY AGE AND SEX..."`), spilled across one cell.
2. **Rows 1-5** (0-indexed) are a *5-row spanning header*. Excel's merged
   cells mean a category label like `"Illiterate"` or `"Below primary"` is
   only physically stored in the top-left cell of its merged block — every
   other cell pandas reads back as `NaN`.
3. **Row 6** is a stray artifact: a single leaked value (`763,638,812`,
   India's total literate population) sitting in an otherwise-empty row
   between the header and the real data. It isn't a real record.
4. **The `Age-group` column mixes types**: ages 7-19 are stored as Excel
   numbers, while `'0-6'`, `'20-24'`, `'All ages'`, `'Age not stated'` are
   strings. Left as-is, this column can't be sorted or filtered consistently.
5. **`Area Name` is inconsistent**: the country row reads `'INDIA'`, while
   every state/UT row is prefixed `'State - '` (e.g. `'State - KERALA'`).
6. **Two constant, uninformative columns**: `Table Name` (always `'C2308'`)
   and `Distt. Code` (always `'000'`, since this table is published at
   state/UT level only, never below).
7. **A fully-empty trailing column** left over from the Excel export.

We fix each of these below.

## 3. Reconstruct the real column headers

The category label for each block of 3 columns (`Persons` / `Males` /
`Females`) is split across up to 3 header rows because of the merged
cells:

- **Row 1** holds the top-level group: `Total`, `Illiterate`, `Literate`, or
  the umbrella label `"Educational level"` (which by itself is not a usable
  category name).
- **Row 2** holds the real sub-category whenever the top-level group is
  `"Educational level"` (e.g. `"Below"`, `"Primary"`, `"Middle"`).
- **Row 3** holds continuation text for a few sub-categories
  (`"Below"` + `"primary"` -> `"Below primary"`).

We forward-fill each row to undo the effect of Excel's cell-merging, being
careful about *how far* to fill:
- Row 1's groups are uneven widths (the `"Educational level"` umbrella
  spans 30 columns), so it gets a plain, unlimited forward-fill.
- Rows 2 and 3 repeat every exactly 3 columns, so we forward-fill with
  `limit=2` — enough to fill out one merged block, but not enough to leak
  into the next one.

In [1]:
def build_column_names(raw):
    meta_cols = ['Table_Name', 'State_Code', 'Distt_Code',
                 'Area_Name', 'Total_Rural_Urban', 'Age_Group']

    top = raw.iloc[1, 6:45].ffill()
    sub2 = raw.iloc[2, 6:45].ffill(limit=2)
    sub3 = raw.iloc[3, 6:45].ffill(limit=2)
    sex_row = raw.iloc[4, 6:45]

    data_cols = []
    for t, s2, s3, sex in zip(top, sub2, sub3, sex_row):
        t = str(t).strip()
        if t in ('Total', 'Illiterate', 'Literate'):
            category = t
        else:
            parts = [p for p in (s2, s3) if pd.notna(p)]
            category = ' '.join(str(p).strip() for p in parts)
        label = f"{category}_{str(sex).strip()}"
        label = "_".join(label.replace('/', '_').split())
        data_cols.append(label)

    return meta_cols + data_cols

cols = build_column_names(raw.dropna(axis=1, how='all'))
print(f'{len(cols)} columns reconstructed, {len(set(cols))} unique')
for c in cols[:10]:
    print(' -', c)
print(' ... (33 more)')


45 columns reconstructed, 45 unique
 - Table_Name
 - State_Code
 - Distt_Code
 - Area_Name
 - Total_Rural_Urban
 - Age_Group
 - Total_Persons
 - Total_Males
 - Total_Females
 - Illiterate_Persons
 ... (33 more)


## 4. Full cleaning pipeline

Now we apply the fix for every problem identified in Section 2.

In [1]:
def clean(path=SRC, sheet=SHEET):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)

    # Problem 7: drop the fully-empty trailing column
    raw = raw.dropna(axis=1, how='all')

    # Problem 2: reconstruct real column names from the spanning header
    raw.columns = build_column_names(raw)

    # Problems 1, 2, 3: drop the title row, the 5-row header block, and the
    # single stray artifact row -> real data starts at Excel row 8
    # (0-based index 7)
    df = raw.iloc[7:].reset_index(drop=True)

    # Problem 4: standardise Age_Group to a single, clean string type
    df['Age_Group'] = df['Age_Group'].apply(
        lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x)
        else str(x).strip()
    )

    # Problem 5: uniform area names (strip the inconsistent "State - " prefix)
    df['Area_Name_Clean'] = (
        df['Area_Name']
        .str.replace(r'^State\s*-\s*', '', regex=True)
        .str.title()
        .replace({'India': 'INDIA'})
    )

    # Problem 6: drop constant, non-informative metadata columns
    constant_cols = [c for c in ['Table_Name', 'Distt_Code']
                     if df[c].nunique(dropna=True) <= 1]
    df = df.drop(columns=constant_cols)

    # Cast every count column to a clean nullable integer type
    non_numeric = ['State_Code', 'Area_Name', 'Area_Name_Clean',
                    'Total_Rural_Urban', 'Age_Group']
    numeric_cols = [c for c in df.columns if c not in non_numeric]
    df[numeric_cols] = df[numeric_cols].apply(
        pd.to_numeric, errors='coerce'
    ).astype('Int64')

    # Sanity checks -- fail loudly if cleaning introduced anything unexpected
    assert df[numeric_cols].isna().sum().sum() == 0, 'Unexpected nulls in count columns'
    assert set(df['Total_Rural_Urban'].unique()) == {'Total', 'Rural', 'Urban'}

    id_cols = ['State_Code', 'Area_Name_Clean', 'Total_Rural_Urban', 'Age_Group']
    return df[id_cols + numeric_cols]

df = clean()
print('Cleaned shape:', df.shape)
print('States/UTs found:', df['Area_Name_Clean'].nunique())


Cleaned shape: (3132, 43)
States/UTs found: 36


## 5. Inspect the cleaned result

In [1]:
print(df.head(8).to_string())


  State_Code Area_Name_Clean Total_Rural_Urban Age_Group  Total_Persons  Total_Males  Total_Females  Illiterate_Persons  Illiterate_Males  Illiterate_Females  Literate_Persons  Literate_Males  Literate_Females  Literate_without_educational_level_Persons  Literate_without_educational_level_Males  Literate_without_educational_level_Females  Below_primary_Persons  Below_primary_Males  Below_primary_Females  Primary_Persons  Primary_Males  Primary_Females  Middle_Persons  Middle_Males  Middle_Females  Matric_Secondary_Persons  Matric_Secondary_Males  Matric_Secondary_Females  Higher_secondary_Intermediate_Pre-University_Senior_secondary_Persons  Higher_secondary_Intermediate_Pre-University_Senior_secondary_Males  Higher_secondary_Intermediate_Pre-University_Senior_secondary_Females  Non-technical_diploma_or_certificate_not_equal_to_degree_Persons  Non-technical_diploma_or_certificate_not_equal_to_degree_Males  Non-technical_diploma_or_certificate_not_equal_to_degree_Females  Technical_dipl

In [1]:
print(df.dtypes.value_counts())


Int64    39
str       4
Name: count, dtype: int64


## 6. Validate against a known figure

India's total 2011 Census population was 1,210,854,977 and total literate
population was 763,638,812 — both widely published figures. If cleaning
introduced an error, this row would no longer match.

In [1]:
check = df.query(
    "Area_Name_Clean == 'INDIA' and Total_Rural_Urban == 'Total' and Age_Group == 'All ages'"
)
print(check[['Total_Persons', 'Literate_Persons']].to_string(index=False))


 Total_Persons  Literate_Persons
    1210854977         763638812


## 7. Save the cleaned data

In [1]:
df.to_csv('C08_clean.csv', index=False)
df.to_excel('C08_clean.xlsx', index=False)
print('Saved C08_clean.csv and C08_clean.xlsx:', df.shape)


Saved C08_clean.csv and C08_clean.xlsx: (3132, 43)


## Summary

| Problem | Fix |
|---|---|
| Multi-row merged header | Reconstructed real column names from rows 1-4, with group-aware forward-fill |
| Title row + stray artifact row | Sliced out; real data starts at row 8 |
| Mixed-type `Age_Group` | Cast to a single clean string type |
| Inconsistent `Area_Name` (`'INDIA'` vs `'State - X'`) | New `Area_Name_Clean` column, prefix stripped and title-cased |
| Constant columns (`Table_Name`, `Distt_Code`) | Dropped |
| Empty trailing column | Dropped |
| Count columns read as mixed/object types | Cast to nullable `Int64` |

Result: cleaned dataset, one row per (State/UT x Rural/Urban/Total
x Age-group), ready for aggregation, joins, or visualization.